In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import functions

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import re

In [2]:
df = pd.read_csv("../../data/clean_train.csv", sep=";",encoding="utf-8-sig", keep_default_na=False)

## Create Transcription Headers

In [3]:
df["transcription_headers_model"] = df["transcription"].apply(functions.extract_headers).apply(functions.headers_for_model)
df["transcription_headers_words"] = df["transcription"].apply(functions.extract_headers).apply(functions.headers_words)
df["transcription_headers_words"][0]


'REASON FOR REFERRAL MEDICATIONS ALLERGIES SOCIAL HISTORY REVIEW OF SYSTEMS PEX EKG ECHOCARDIOGRAM (FOR SYNCOPE) IMPRESSION RECOMMENDATIONS'

# TF-IDF

### Train/Test Split

In [4]:
from sklearn.model_selection import train_test_split

train_indices, valid_indices = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["medical_specialty"],
)

y_train = df.loc[train_indices, "medical_specialty"]
y_valid = df.loc[valid_indices, "medical_specialty"]

### Classifiers

In [5]:
linear_svc = LinearSVC(C=0.5,class_weight="balanced",)
logistic_regression = LogisticRegression(C=1.0,class_weight="balanced",max_iter=2000,random_state=42,)
multinomial_nb = MultinomialNB()

### Testing Baseline

In [6]:
input_just_description = functions.build_input_variants(df, ["description"])

pd.DataFrame([functions.evaluate_input_extra(input_just_description, "Just description", multinomial_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1]])

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Multinomial NB,Just description,0.384615,0.132751,0.0,Dermatology


### Testing Each Field

In [7]:
input1 = functions.build_input_variants(df, ["description"])
input2 = functions.build_input_variants(df, ["sample_name"])
input3 = functions.build_input_variants(df, ["transcription"])
input4 = functions.build_input_variants(df, ["keywords"])
input5 = functions.build_input_variants(df, ["transcription_headers_model"])
input6 = functions.build_input_variants(df, ["transcription_headers_words"])


#results = pd.DataFrame([
#    functions.evaluate_input_extra(input1, "Description", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input2, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input3, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input4, "keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input5, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input6, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#])

#results

### Testing 2 Fields Combinations

#### Description

In [8]:
input1 = functions.build_input_variants(df, ["description","sample_name"])
input3 = functions.build_input_variants(df, ["description","transcription"])
input4 = functions.build_input_variants(df, ["description","keywords"])
input5 = functions.build_input_variants(df, ["description","transcription_headers_model"])
input6 = functions.build_input_variants(df, ["description","transcription_headers_words"])


#results = pd.DataFrame([
#    functions.evaluate_input_extra(input1, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input3, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input4, "keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input5, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input6, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#])

#results

#### Sample Name

In [9]:
input1 = functions.build_input_variants(df,["description","sample_name"])
input2 = functions.build_input_variants(df, ["sample_name","transcription"])
input3 = functions.build_input_variants(df, ["keywords","sample_name"])
input4 = functions.build_input_variants(df, ["sample_name","transcription_headers_model"])
input5 = functions.build_input_variants(df, ["sample_name","transcription_headers_words"])


#results = pd.DataFrame([
#    functions.evaluate_input_extra(input1, "description", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input2, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input3, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input4, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#    functions.evaluate_input_extra(input5, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
#])

#results

#### Keywords

In [10]:
input1 = functions.build_input_variants(df,["description","keywords"])
input2 = functions.build_input_variants(df, ["keywords","transcription"])
input3 = functions.build_input_variants(df, ["keywords","sample_name"])
input4 = functions.build_input_variants(df, ["keywords","transcription_headers_model"])
input5 = functions.build_input_variants(df, ["keywords","transcription_headers_words"])


results = pd.DataFrame([
    functions.evaluate_input_extra(input1, "description", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input2, "transcription", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input3, "sample name", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input4, "transcription headers model", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input5, "transcription headers words", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
])

results

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,description,0.831144,0.772146,0.400000,Psychiatry-Psychology
1,Linear SVC,transcription,0.497186,0.534895,0.153846,Neurosurgery
2,Linear SVC,sample name,0.889306,0.864031,0.777778,Psychiatry-Psychology
3,Linear SVC,transcription headers model,0.879925,0.844047,0.705882,Psychiatry-Psychology
4,Linear SVC,transcription headers words,0.883677,0.848029,0.705882,Psychiatry-Psychology


### Keywords and Transcription Headers

Linear SVC 

Logistic Regression

Multinomial Naive Bayes

In [11]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_key = functions.build_input_variants(df, ["keywords"])

#pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred2, result2 = functions.evaluate_input_extra(input_key, "key", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred3, result3 = functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
#pred4, result4 = functions.evaluate_input_extra(input_key, "key", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
#pred5, result5 = functions.evaluate_input_extra(input, "key and Headers", multinomial_nb, "multinomial_nb", train_indices, valid_indices, y_train, y_valid)
#pred6, result6 = functions.evaluate_input_extra(input_key, "key", multinomial_nb, "multinomial_nb", train_indices, valid_indices, y_train, y_valid)

#results = pd.DataFrame([result1, result2, result3, result4, result5, result6])
#results

In [12]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_desc = functions.build_input_variants(df, ["description", "keywords", "transcription_headers_model"])
input_desc_sample = functions.build_input_variants(df, ["description", "sample_name", "transcription_headers_model"])
input_sample = functions.build_input_variants(df, ["sample_name", "keywords"])


#pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred2, result2 = functions.evaluate_input_extra(input_desc, "desc, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred3, result3 = functions.evaluate_input_extra(input_desc_sample, "desc, sample, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred4, result4 = functions.evaluate_input_extra(input_sample, "sample, key and Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
#pred5, result5 = functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#pred6, result6 = functions.evaluate_input_extra(input_desc, "desc, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#pred7, result7 = functions.evaluate_input_extra(input_desc_sample, "desc, sample, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#pred8, result8 = functions.evaluate_input_extra(input_sample, "sample, key and Headers", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
#results = pd.DataFrame([result1, result2, result3, result4, result5, result6, result7, result8])
#results

In [24]:
input_keywords = functions.build_input_variants(df, ["description", "keywords", "sample_name", "transcription_headers_model"])
input = functions.build_input_variants(df, ["description", "sample_name", "transcription_headers_model"])
input_only_keywords = functions.build_input_variants(df, ["keywords"])

predict_keywords, result_keywords = functions.evaluate_input_extra(input_keywords, "With Keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
predict, result = functions.evaluate_input_extra(input, "Without Keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
predict_only_keywords, result_only_keywords = functions.evaluate_input_extra(input_only_keywords, "Only Keywords", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)
predict_keywords2, result_keywords2 = functions.evaluate_input_extra(input_keywords, "With Keywords", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
predict2, result2 = functions.evaluate_input_extra(input, "Without Keywords", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)
predict_only_keywords2, result_only_keywords2 = functions.evaluate_input_extra(input_only_keywords, "Only Keywords", logistic_regression, "logistic regression", train_indices, valid_indices, y_train, y_valid)

results = pd.DataFrame([result_keywords, result, result_only_keywords, result_keywords2, result2, result_only_keywords2])
results


,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,With Keywords,0.819887,0.794207,0.651163,Neurosurgery
1,Linear SVC,Without Keywords,0.362101,0.414901,0.102564,Neurosurgery
2,Linear SVC,Only Keywords,0.840525,0.822564,0.166667,Psychiatry-Psychology
3,logistic regression,With Keywords,0.694184,0.692605,0.400000,Neurosurgery
4,logistic regression,Without Keywords,0.452158,0.495792,0.232558,Neurosurgery
5,logistic regression,Only Keywords,0.782364,0.764078,0.166667,Psychiatry-Psychology


In [14]:
labels = sorted(y_valid.unique())
predict_combined = np.where(df.loc[valid_indices, "keywords"] == "", predict2, predict_only_keywords)

functions.show_classification_report(y_valid,predict_combined,labels,)

                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.90      0.90      0.90        62
             Dermatology       1.00      0.80      0.89         5
        Gastroenterology       1.00      0.82      0.90        39
        General Medicine       0.65      0.95      0.77        42
               Neurology       0.78      0.76      0.77        37
            Neurosurgery       0.88      0.88      0.88        16
   Obstetrics-Gynecology       0.96      0.85      0.90        27
           Ophthalmology       0.83      1.00      0.91        15
              Orthopedic       0.90      0.93      0.92        60
   Psychiatry-Psychology       0.75      0.75      0.75         8
               Radiology       0.97      0.89      0.93        44
                 Surgery       0.99      0.94      0.97       178

                accuracy                           0.90       533
               macro avg       0.88      0.87      0.87       533
        

In [15]:
for x in range (0,6):   
    sem_kw = (df.loc[valid_indices, "keywords"] == "").to_numpy()
    pred = [predict_keywords, predict,  predict_only_keywords, predict_keywords2, predict2, predict_only_keywords2][x]
    certo = (pred == y_valid.to_numpy())
    print("com keywords:", certo[~sem_kw].mean(), "| sem keywords:", certo[sem_kw].mean())

com keywords: 0.899103139013453 | sem keywords: 0.41379310344827586
com keywords: 0.336322869955157 | sem keywords: 0.4482758620689655
com keywords: 0.9887892376681614 | sem keywords: 0.08045977011494253
com keywords: 0.7354260089686099 | sem keywords: 0.4827586206896552
com keywords: 0.4484304932735426 | sem keywords: 0.45977011494252873
com keywords: 0.9192825112107623 | sem keywords: 0.08045977011494253
